In [1]:
import pandas as pd

df = pd.read_csv("Data/viirs-snpp_2024_India.csv")
print(df.shape)
print(df.head())

(552312, 15)
   latitude  longitude  bright_ti4  scan  track    acq_date  acq_time  \
0  17.16963   79.99036      332.05  0.63   0.72  2024-01-01       702   
1  18.24348   83.95551      333.71  0.33   0.56  2024-01-01       702   
2  18.74076   83.98029      340.18  0.32   0.55  2024-01-01       702   
3  18.70938   83.45336      327.85  0.35   0.57  2024-01-01       702   
4  18.70869   83.45004      329.70  0.35   0.57  2024-01-01       702   

  satellite instrument confidence  version  bright_ti5   frp daynight  type  
0         N      VIIRS          n        2      294.54  3.68        D     0  
1         N      VIIRS          n        2      296.66  1.75        D     0  
2         N      VIIRS          n        2      299.86  4.24        D     0  
3         N      VIIRS          n        2      299.34  2.87        D     0  
4         N      VIIRS          n        2      296.81  2.04        D     0  


In [31]:
print(df.columns.tolist())
# Should show: ['latitude', 'longitude', 'acq_date', 'confidence', 'bright_ti4', 'frp'

['latitude', 'longitude', 'acq_date', 'confidence', 'region']


In [33]:
df = df[['latitude', 'longitude', 'acq_date', 'confidence', 'bright_ti4', 'frp']]
df['acq_date'] = pd.to_datetime(df['acq_date'])

KeyError: "['bright_ti4', 'frp'] not in index"

In [3]:
df = df[df['acq_date'] >= '2024-07-01']

In [32]:
import h3

df['region'] = df.apply(
    lambda r: h3.latlng_to_cell(r['latitude'], r['longitude'], 4),
    axis=1
)

In [5]:
df_grouped = df.groupby(['region', 'acq_date']).size().reset_index(name='fires')
df_grouped['label'] = (df_grouped['fires'] > 0).astype(int)

In [6]:
all_regions = df_grouped['region'].unique()
all_dates = pd.date_range(df['acq_date'].min(), df['acq_date'].max())

full_index = pd.MultiIndex.from_product([all_regions, all_dates], names=['region', 'acq_date'])
df_full = pd.DataFrame(index=full_index).reset_index()
df_full = df_full.merge(df_grouped[['region', 'acq_date', 'label']],
                        on=['region', 'acq_date'],
                        how='left')

df_full['label'] = df_full['label'].fillna(0)

In [7]:
df_full.shape

(292008, 3)

In [8]:
active_regions = df_grouped['region'].unique()

In [9]:
all_dates = pd.date_range(df['acq_date'].min(), df['acq_date'].max())

full_index = pd.MultiIndex.from_product(
    [active_regions, all_dates],
    names=['region', 'acq_date']
)

df_full = pd.DataFrame(index=full_index).reset_index()

df_full = df_full.merge(
    df_grouped[['region', 'acq_date', 'label']],
    on=['region', 'acq_date'],
    how='left'
)

df_full['label'] = df_full['label'].fillna(0)

In [10]:
df_full.shape

(292008, 3)

In [11]:

# Unique regions (already fire-active regions)
regions = df_grouped['region'].unique()

# Date range
dates = pd.date_range(df['acq_date'].min(), df['acq_date'].max())

# Create grid
full_index = pd.MultiIndex.from_product(
    [regions, dates],
    names=['region', 'acq_date']
)

df_full = pd.DataFrame(index=full_index).reset_index()

# Merge labels
df_full = df_full.merge(
    df_grouped[['region', 'acq_date', 'label']],
    on=['region', 'acq_date'],
    how='left'
)

# Fill missing = no fire
df_full['label'] = df_full['label'].fillna(0)

In [12]:
df_full.shape

(292008, 3)

In [13]:
len(df_grouped['region'].unique())

1587

In [14]:
top_region = (
    df_grouped.groupby('region')['fires'].sum().sort_values(ascending=False).head(4000).index
)

df_grouped = df_grouped[df_grouped['region'].isin(top_region)]

In [15]:
df_full.shape

(292008, 3)

In [16]:
df_full.columns

Index(['region', 'acq_date', 'label'], dtype='object')

In [17]:
df_full.head()

,region,acq_date,label
0,84209a3ffffffff,2024-07-01,0.0
1,84209a3ffffffff,2024-07-02,0.0
2,84209a3ffffffff,2024-07-03,0.0
3,84209a3ffffffff,2024-07-04,0.0
4,84209a3ffffffff,2024-07-05,0.0


In [18]:
df_full['label'].value_counts()

label
0.0    264548
1.0     27460
Name: count, dtype: int64

In [24]:
df_full['region_enc'] = df_full['region'].astype('category').cat.codes

In [25]:
df_full = df_full.sort_values('acq_date')

split_date = df_full['acq_date'].quantile(0.8)

train = df_full[df_full['acq_date'] <= split_date]
test = df_full[df_full['acq_date'] > split_date]

In [26]:
train['region_enc'] = train['region'].astype('category').cat.codes
test['region_enc'] = test['region'].astype('category').cat.codes

X_train = train[['region_enc']]
y_train = train['label']

X_test = test[['region_enc']]
y_test = test['label']

C:\Users\vaiks\AppData\Local\Temp\ipykernel_3612\626472019.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['region_enc'] = train['region'].astype('category').cat.codes
C:\Users\vaiks\AppData\Local\Temp\ipykernel_3612\626472019.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['region_enc'] = test['region'].astype('category').cat.codes


In [27]:
test['region_enc'] = test['region'].map(
    dict(zip(train['region'], train['region_enc']))
).fillna(-1)

C:\Users\vaiks\AppData\Local\Temp\ipykernel_3612\4269750058.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['region_enc'] = test['region'].map(


In [28]:
from lightgbm import LGBMClassifier

model = LGBMClassifier()
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 15450, number of negative: 219426
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000624 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 234876, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.065779 -> initscore=-2.653406
[LightGBM] [Info] Start training from score -2.653406


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [29]:
from sklearn.metrics import roc_auc_score, classification_report

y_prob = model.predict_proba(X_test)[:1]
y_pred = model.predict(X_test)

print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

ValueError: Found input variables with inconsistent numbers of samples: [57132, 1]